# Python optimization: batch order pricing

A business needs to recalculate a batch of order totals when the discount, tax rate, or
shipping fee changes. Start with a plain Python loop, speed up the same calculation,
then expose it through REST and WebSocket APIs.

1. **CPU:** ordinary Python → NumPy → Numba JIT.
2. **GPU:** CUDA ufunc → custom kernel → device-memory reuse.
3. **Interpreter:** Python threads with the GIL on and off.
4. **API:** four standalone FastAPI files, measured with Python Locust.


## Before running

Use a regular Python 3.13 or 3.14 environment with `requirements.txt`, or
`requirements-cuda.txt` for the NVIDIA GPU. The GIL experiment launches a separate
free-threaded interpreter; it uses only the standard library.

Run from this folder. Part 4 starts temporary local servers and runs Locust, then stops
the servers. Set `RUN_API_DEMO = False` to skip it. See [README.md](README.md) for setup.

**Batch-size experiment:** edit `BATCH_SIZE` in the next cell. Run Parts 1–2 once to load
the functions. For each new size, rerun the setting cell and the comparison call immediately
before Part 3. Each backend will process exactly that many orders.

In [ ]:
# Suggested sizes: 20_000, 200_000, 2_000_000, 10_000_000
BATCH_SIZE = 10_000_000

In [ ]:
import os
import sys
from pathlib import Path
from importlib.metadata import version

import numpy as np
from numpy import testing
from numba import njit
from IPython.display import Markdown, display
import matplotlib.pyplot as plt

ROOT = Path.cwd()
assert (ROOT / 'gil_benchmark.py').exists(), 'Open this notebook from the demo folder.'
RUN_API_DEMO = os.environ.get('PRICING_RUN_API', '1') == '1'
API_BACKEND = os.environ.get('PRICING_BACKEND', 'numba')  # Try numpy or cuda too.
print(f'Python {sys.version.split()[0]} | NumPy {version("numpy")} | Numba {version("numba")}')
print(f'{BATCH_SIZE:,} orders for CPU/GPU benchmarks; API demo: {RUN_API_DEMO}; optimized backend: {API_BACKEND}')

## Part 1 — Calculate order totals on the CPU

Each order has a unit price and quantity. Apply a batch-wide discount, then tax,
then a flat shipping fee **per order**:

$$\text{total} = (\text{unit price} \times \text{quantity})
                \times (1-\text{discount}) \times (1+\text{tax rate})
                + \text{shipping fee}.$$

For example: 2 items at $10, a 10% discount, 10% tax, and $5 shipping gives **$24.80**.
Every order is independent, making the same business rule suitable for CPU and GPU implementations.

Generate the seeded batch once, outside the timed regions. Arrays use `float32` for the
numerical demonstration; a billing system would also define its monetary rounding policy.

In [ ]:
def make_orders(n):
    rng = np.random.default_rng(7)
    return (
        rng.uniform(1, 100, n).astype(np.float32),
        rng.integers(1, 21, n).astype(np.float32),  # Whole quantities, stored as numeric arrays.
    )

orders = make_orders(BATCH_SIZE)
unit_prices, quantities = orders
parameters = tuple(np.float32(v) for v in (0.1, 0.08, 5))
print('Input storage:', sum(a.nbytes for a in orders) / 1e6, 'MB')

### Ordinary Python: start with the business rule

The loop multiplies the price by the quantity, applies the discount and tax, and adds
shipping. Converting each input element with `float()` keeps the loop's math in ordinary
Python floats. Each version returns one total per order in a `float32` host array.

In [ ]:
def price_python(unit_prices, quantities, discount, tax_rate, shipping_fee):
    discount, tax_rate, shipping_fee = float(discount), float(tax_rate), float(shipping_fee)
    out = np.empty(len(unit_prices), dtype=np.float32)
    for i in range(len(unit_prices)):
        subtotal = float(unit_prices[i]) * float(quantities[i])
        discounted = subtotal * (1.0 - discount)
        out[i] = discounted * (1.0 + tax_rate) + shipping_fee
    return out

In [ ]:
# Check a hand-calculated example before benchmarking a large batch.
example = (np.array([10], dtype=np.float32), np.array([2], dtype=np.float32))
testing.assert_allclose(price_python(*example, 0.1, 0.1, 5), [24.8], rtol=2e-6)
reference = price_python(*orders, *parameters)
assert reference.shape == (BATCH_SIZE,) and reference.dtype == np.float32
assert np.all(np.isfinite(reference)) and np.all(reference >= 0)
print(f'Checked {BATCH_SIZE:,} reference order totals.')

In [ ]:
python_timing = %timeit -o -r 3 -n 3 price_python(*orders, *parameters)

### NumPy: apply the rule to whole arrays

**Predict:** which work moves out of Python? NumPy runs the arithmetic loops in native
code, but these expressions still allocate intermediate arrays and make multiple passes
through memory. `np.vectorize` would still repeatedly call Python; that is a different tool.

In [ ]:
def price_numpy(unit_prices, quantities, discount, tax_rate, shipping_fee):
    subtotal = unit_prices * quantities
    discounted = subtotal * (np.float32(1.0) - discount)
    return discounted * (np.float32(1.0) + tax_rate) + shipping_fee

In [ ]:
numpy_result = price_numpy(*orders, *parameters)
testing.assert_allclose(numpy_result, reference, rtol=2e-6, atol=1e-3)
print('NumPy agrees with the Python reference.')

In [ ]:
numpy_timing = %timeit -o -r 3 -n 10 price_numpy(*orders, *parameters)

### Numba: compile the same loop

`@njit` compiles the numerical function into native CPU code. The first call pays for
compilation; later calls reuse the specialization. The loop body is the same as our
Python baseline, and the intermediate arrays from the NumPy version disappear.

`nogil=True` lets the compiled function release the GIL while it runs. It does not create
threads or disable the GIL throughout the interpreter.

**Predict:** will this compiled loop beat the NumPy expressions for this batch size?

In [ ]:
@njit(nogil=True)
def price_numba(unit_prices, quantities, discount, tax_rate, shipping_fee):
    discount, tax_rate, shipping_fee = float(discount), float(tax_rate), float(shipping_fee)
    out = np.empty(len(unit_prices), dtype=np.float32)
    for i in range(len(unit_prices)):
        subtotal = float(unit_prices[i]) * float(quantities[i])
        discounted = subtotal * (1.0 - discount)
        out[i] = discounted * (1.0 + tax_rate) + shipping_fee
    return out

In [ ]:
# First call includes compilation for this signature; rerunning this cell is a warm call.
%time numba_result = price_numba(*orders, *parameters)
testing.assert_allclose(numba_result, reference, rtol=2e-6, atol=1e-3)
print('Compiled signatures:', price_numba.signatures)

In [ ]:
numba_timing = %timeit -o -r 3 -n 10 price_numba(*orders, *parameters)

In [ ]:
def timing_table(rows):
    lines = ['| Implementation | Mean ms | Speedup vs Python |', '|---|---:|---:|']
    for name, seconds in rows:
        lines.append(f'| {name} | {seconds * 1000:.3f} | {python_timing.average / seconds:.2f}× |')
    display(Markdown('\n'.join(lines)))

cpu_rows = [('Python', python_timing.average), ('NumPy', numpy_timing.average),
            ('Numba (warm)', numba_timing.average)]
timing_table(cpu_rows)

### Try a different pricing scenario

Change the discount, tax rate, and shipping fee, then check that all three versions agree.
To compare batch sizes, use the **Rerun the CPU/GPU comparison** cell at the end of Part 2.
It rebuilds the inputs and measures every implementation using the same batch size.

## Part 2 — Calculate the same totals on the GPU

Follow the CUDA course's progression: start with a CUDA ufunc, then write a custom
kernel, then manage device arrays explicitly. The business rule stays the same.
NVIDIA's maintained `numba-cuda` package still uses `from numba import cuda`.

There is little arithmetic per order. **Predict:** will transferring a batch to the GPU
cost more than the GPU saves in computation? We will measure that rather than assume
that the GPU is fastest.

GPU launches are asynchronous. Our kernel timers explicitly wait for completion.
Kernel-only timing and timing that returns a host array are different measurements.

In [ ]:
from numba import cuda, vectorize

CUDA_AVAILABLE = os.environ.get('PRICING_SKIP_CUDA', '0') != '1' and cuda.is_available()
if CUDA_AVAILABLE:
    print('CUDA device:', cuda.get_current_device())
else:
    print('CUDA skipped. Install requirements-cuda.txt and check nvidia-smi.')

### A CUDA ufunc

Write the rule for one order; the decorator applies it across arrays.
Passing host arrays includes implicit device allocation and copies.

In [ ]:
if CUDA_AVAILABLE:
    @vectorize(['float32(float32, float32, float32, float32, float32)'], target='cuda')
    def price_ufunc(unit_price, quantity, discount, tax_rate, shipping_fee):
        subtotal = unit_price * quantity
        discounted = subtotal * (np.float32(1.0) - discount)
        return discounted * (np.float32(1.0) + tax_rate) + shipping_fee

    ufunc_result = price_ufunc(*orders, *parameters)
    testing.assert_allclose(ufunc_result, reference, rtol=2e-6, atol=1e-3)

In [ ]:
if CUDA_AVAILABLE:
    ufunc_timing = %timeit -o -r 3 -n 5 price_ufunc(*orders, *parameters)

### A custom kernel with a grid-stride loop

`cuda.grid(1)` selects a thread's starting order. `cuda.gridsize(1)` is the stride to
its next order. This handles the tail of the batch and batches larger than the grid.
A kernel writes its results into an output array instead of returning them.

**Predict:** what happens if we launch this entire calculation with just one GPU thread?

In [ ]:
if CUDA_AVAILABLE:
    @cuda.jit
    def price_kernel(unit_prices, quantities, discount, tax_rate, shipping_fee, out):
        start = cuda.grid(1)
        stride = cuda.gridsize(1)
        for i in range(start, unit_prices.size, stride):
            subtotal = unit_prices[i] * quantities[i]
            discounted = subtotal * (np.float32(1.0) - discount)
            out[i] = discounted * (np.float32(1.0) + tax_rate) + shipping_fee

    device_orders = tuple(cuda.to_device(a) for a in orders)
    device_output = cuda.device_array(BATCH_SIZE, dtype=np.float32)
    threads_per_block = 128
    blocks = min(1024, (BATCH_SIZE + threads_per_block - 1) // threads_per_block)

    # Compile separately, then wait before measuring warm launches.
    %time price_kernel[blocks, threads_per_block](*device_orders, *parameters, device_output); cuda.synchronize()
    testing.assert_allclose(device_output.copy_to_host(), reference, rtol=2e-6, atol=1e-3)

In [ ]:
if CUDA_AVAILABLE:
    # Intentionally inefficient configuration, timed once like the course.
    %time price_kernel[1, 1](*device_orders, *parameters, device_output); cuda.synchronize()
    testing.assert_allclose(device_output.copy_to_host(), reference, rtol=2e-6, atol=1e-3)

In [ ]:
if CUDA_AVAILABLE:
    def launch_only():
        price_kernel[blocks, threads_per_block](*device_orders, *parameters, device_output)
        cuda.synchronize()

    kernel_timing = %timeit -o -r 3 -n 10 launch_only()

### Include the copies

The kernel-only measurement leaves all inputs and outputs on the GPU. Our CPU functions
return a host array. For a fairer comparison, time GPU functions that do the same:

- **Full round trip:** copy the batch to the GPU, calculate, copy totals back.
- **Persistent batch:** upload the batch once, then calculate and copy totals back for
  each new discount/tax/shipping scenario.

The persistent-batch version matches the CUDA API. It still copies every total back and
summarizes them on the CPU. Reducing the results on the GPU would be a further optimization.

In [ ]:
if CUDA_AVAILABLE:
    def price_cuda_roundtrip():
        inputs = tuple(cuda.to_device(a) for a in orders)
        output = cuda.device_array(BATCH_SIZE, dtype=np.float32)
        price_kernel[blocks, threads_per_block](*inputs, *parameters, output)
        return output.copy_to_host()  # Blocking copy waits for completion.

    def price_cuda_persistent():
        price_kernel[blocks, threads_per_block](*device_orders, *parameters, device_output)
        return device_output.copy_to_host()

    testing.assert_allclose(price_cuda_roundtrip(), reference, rtol=2e-6, atol=1e-3)
    testing.assert_allclose(price_cuda_persistent(), reference, rtol=2e-6, atol=1e-3)

In [ ]:
if CUDA_AVAILABLE:
    roundtrip_timing = %timeit -o -r 3 -n 5 price_cuda_roundtrip()
    persistent_timing = %timeit -o -r 3 -n 5 price_cuda_persistent()

In [ ]:
if CUDA_AVAILABLE:
    timing_table(cpu_rows + [
        ('CUDA ufunc, implicit copies', ufunc_timing.average),
        ('CUDA custom, full round trip', roundtrip_timing.average),
        ('CUDA custom, persistent batch + host output', persistent_timing.average),
    ])
    print(f'Kernel only, resident input/output: {kernel_timing.average * 1000:.3f} ms (different contract)')

### Rerun the CPU/GPU comparison

After the first walkthrough, change only **`BATCH_SIZE`** in the setting cell near the top.
Run that setting cell, then the comparison call below. You do not need to rerun the
walkthrough's individual timing cells. Each comparison creates fresh inputs and GPU
buffers, checks the answers, warms compilation, and repeats every measurement three times.

Try **20,000 → 200,000 → 2,000,000 → 10,000,000** orders. The chart shows milliseconds
per batch: shorter bars mean faster calculations. The table also reports million orders
per second, so throughput is comparable across batch sizes. Previous sizes remain in the
history table until you restart the kernel.

The chart compares functions returning a host array. GPU kernel-only timing is listed
separately because it leaves the result on the device. With this simple arithmetic,
copying data can dominate; increasing the batch size does not guarantee a GPU win over
optimized NumPy or Numba. Compare those timings as well as the plain Python baseline.

Batch size here controls Parts 1–2. The GIL/API sections keep their own demonstration
size, so a large GPU experiment does not also create a large threaded or load-test run.

In [ ]:
from statistics import median
from timeit import repeat

# Keep results across reruns; restart the kernel to clear this history.
if 'batch_history' not in globals():
    batch_history = {}


def compare_batch(batch_size):
    if type(batch_size) is not int or batch_size < 1:
        raise ValueError('BATCH_SIZE must be a positive integer.')
    batch = make_orders(batch_size)
    args = parameters
    expected = price_python(*batch, *args)
    rows = []

    def measure(name, calculate, returns_host=True):
        # Warm compilation and check correctness outside the timed region.
        result = calculate()
        if returns_host:
            testing.assert_allclose(result, expected, rtol=2e-6, atol=1e-3)
        seconds = median(repeat(calculate, repeat=3, number=1))
        rows.append((name, seconds, returns_host))

    measure('Python', lambda: price_python(*batch, *args))
    measure('NumPy', lambda: price_numpy(*batch, *args))
    measure('Numba', lambda: price_numba(*batch, *args))

    if CUDA_AVAILABLE:
        measure('CUDA ufunc + copies', lambda: price_ufunc(*batch, *args))
        device_batch = tuple(cuda.to_device(a) for a in batch)
        output = cuda.device_array(batch_size, dtype=np.float32)
        grid = min(1024, (batch_size + 127) // 128)

        def persistent():
            price_kernel[grid, 128](*device_batch, *args, output)
            return output.copy_to_host()  # Includes completion and the output copy.

        def roundtrip():
            inputs = tuple(cuda.to_device(a) for a in batch)
            result = cuda.device_array(batch_size, dtype=np.float32)
            price_kernel[grid, 128](*inputs, *args, result)
            return result.copy_to_host()

        def kernel_only():
            price_kernel[grid, 128](*device_batch, *args, output)
            cuda.synchronize()  # Time completed work, rather than just dispatch.

        measure('CUDA full round trip', roundtrip)
        measure('CUDA persistent + output copy', persistent)
        kernel_only()
        testing.assert_allclose(output.copy_to_host(), expected, rtol=2e-6, atol=1e-3)
        measure('CUDA kernel only (device output)', kernel_only, returns_host=False)

    results = {name: seconds for name, seconds, _ in rows}
    batch_history[batch_size] = results
    print(f'{batch_size:,} orders — correctness checks passed; median of 3 warm runs.')
    lines = ['| Implementation | Batch time (ms) | Million orders/s |', '|---|---:|---:|']
    for name, seconds, _ in rows:
        lines.append(f'| {name} | {seconds * 1000:.3f} | {batch_size / seconds / 1e6:.2f} |')
    display(Markdown('\n'.join(lines)))

    host_rows = [(name, seconds) for name, seconds, host in rows if host]
    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh([name for name, _ in host_rows], [seconds * 1000 for _, seconds in host_rows],
                   color=['#64748b' if name in {'Python', 'NumPy', 'Numba'} else '#0d9488'
                          for name, _ in host_rows])
    ax.invert_yaxis()
    ax.set_xscale('log')  # Keeps fast native implementations visible beside Python.
    ax.bar_label(bars, labels=[f'{seconds * 1000:.3f} ms' for _, seconds in host_rows], padding=4)
    left, right = ax.get_xlim()
    ax.set_xlim(left, right * 3)
    ax.set(xlabel='Milliseconds per batch — logarithmic scale; lower is faster',
           title=f'CPU vs GPU: {batch_size:,} orders, host output included')
    fig.tight_layout()
    plt.show()

    columns = [('Python', 'Python ms'), ('NumPy', 'NumPy ms'), ('Numba', 'Numba ms'),
               ('CUDA persistent + output copy', 'GPU + output copy ms'),
               ('CUDA kernel only (device output)', 'GPU kernel only ms')]
    history = ['| Orders | ' + ' | '.join(title for _, title in columns) + ' |',
               '|---:|' + '---:|' * len(columns)]
    for count, previous in sorted(batch_history.items()):
        values = [f'{previous[name] * 1000:.3f}' if name in previous else '—' for name, _ in columns]
        history.append(f'| {count:,} | ' + ' | '.join(values) + ' |')
    display(Markdown('**Batch-size history**\n\n' + '\n'.join(history)))
    return results

In [ ]:
# After changing BATCH_SIZE at the top, rerun this cell to compare the new load.
batch_results = compare_batch(BATCH_SIZE)

## Part 3 — Python threads with the GIL on and off

The GIL is the **global interpreter lock**. A free-threaded Python build can disable it;
a regular Python build cannot gain that capability through a runtime flag. Python 3.14t
is recommended for this experiment; a compatible 3.15t also works for this stdlib-only helper.

Run the **same free-threaded executable** in fresh processes with `-X gil=1` and `-X gil=0`.
Each process compares serial and threaded pricing of an equal-sized batch and verifies
its actual GIL state. Every worker owns its accumulator and reads immutable orders.

`gil_benchmark.py` uses the same pricing rule with a separately seeded stdlib fixture.
It imports no NumPy or Numba, keeping native code that can already release the GIL out
of this comparison. Disabling the GIL alone does not parallelize a serial loop.

In [ ]:
import json
import shutil
import subprocess

# Prefer the free-threaded environment in this workspace; allow an explicit override.
local_ft = ROOT / '.venv-free-threaded' / 'bin' / 'python'
FT_PYTHON = (os.environ.get('PRICING_FT_PYTHON')
             or (str(local_ft) if local_ft.is_file() else None)
             or shutil.which('python3.14t') or shutil.which('python3.15t'))
GIL_BATCH_SIZE = int(os.environ.get('PRICING_COUNT', '200000'))  # Separate from the CPU/GPU experiment.
GIL_WORKERS = min(4, os.cpu_count() or 1, GIL_BATCH_SIZE)
if FT_PYTHON:
    print('Free-threaded executable:', FT_PYTHON)
else:
    print('GIL comparison skipped: set PRICING_FT_PYTHON to a free-threaded Python executable.')

In [ ]:
gil_results = []
if FT_PYTHON:
    for gil in (1, 0):
        command = [FT_PYTHON, '-X', f'gil={gil}', str(ROOT / 'gil_benchmark.py'),
                   '--expect-gil', str(gil), '--count', str(GIL_BATCH_SIZE),
                   '--workers', str(GIL_WORKERS), '--repeat', '3']
        completed = subprocess.run(command, check=True, capture_output=True, text=True, timeout=180)
        result = json.loads(completed.stdout)
        gil_results.append(result)
        print(f'GIL {gil}: serial {result["serial_seconds"]:.4f}s; '
              f'{GIL_WORKERS} threads {result["threaded_seconds"]:.4f}s; '
              f'{result["thread_speedup"]:.2f}× within-mode speedup')
    assert gil_results[0]['total_amount'] == gil_results[1]['total_amount']

In [ ]:
if gil_results:
    labels, seconds = [], []
    for result in gil_results:
        label = 'GIL on' if result['gil_enabled'] else 'GIL off'
        labels += [label + ' / serial', label + f' / {GIL_WORKERS} threads']
        seconds += [result['serial_seconds'], result['threaded_seconds']]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.barh(labels, np.array(seconds) * 1000, color=['#64748b', '#64748b', '#0d9488', '#0d9488'])
    ax.set(xlabel='Median elapsed ms (lower is better)', title='Same Python work, same free-threaded executable')
    plt.show()

**Discuss:** did turning off the GIL change serial performance? Did threads help in both
modes? Account for thread overhead and available CPU cores when reading the timings.
The helper warms every pool thread before timing and checks identical chunk results.

This is an interpreter experiment. It does not establish that every extension in the
FastAPI environment supports free threading. Numba's `nogil=True` is a separate mechanism.

## Part 4 — Expose pricing through REST and WebSockets

A pricing tool sends a discount, tax rate, and shipping fee. The server recalculates a
fixed batch and returns its total amount, average order total, and a small preview.

| File | Route | Calculation |
|---|---|---|
| `rest_unoptimized.py` | `POST /quote` | Python loop |
| `rest_optimized.py` | `POST /quote` | NumPy, Numba (default), or CUDA |
| `websocket_unoptimized.py` | `WS /quote` | Same Python loop |
| `websocket_optimized.py` | `WS /quote` | Same selected optimized implementation |

Each file is standalone. Within each REST/WebSocket pair, the request model, startup,
pricing logic, and response are identical. The WebSocket adapter keeps the connection
open so a client can submit updated pricing settings and receive another quote.

In [ ]:
# Display the actual small transport adapters, not a second implementation.
for name in ('rest_unoptimized.py', 'websocket_unoptimized.py'):
    source = (ROOT / name).read_text()
    print(f'\n{name}\n' + source[source.index('# Transport adapter:'):])

Both adapters call `await run_in_threadpool(process, pricing)` to keep CPU work off the
event-loop thread. Ordinary Python threads still contend for the GIL; Numba releases it
while executing the compiled calculation.

CPU output arrays belong to each request. The CUDA version keeps its inputs and one
output buffer on the GPU, with a lock covering calculation and output copy so concurrent
requests cannot mix results. This simple design serializes access to that GPU buffer.

`compute_ms` measures pricing and summary work inside the worker, including GPU waits and
copies. It excludes request validation, worker-pool queueing, JSON serialization, and the
network. Locust measures the full client round trip.

### Use Python Locust to measure the API

`RestUser` and `WebSocketUser` send the same sequence of pricing settings. REST reuses
HTTP connections; WebSocket keeps a connection open. Every WebSocket quote reports a
Locust request event, including failures; handshakes have a separate `/connect` statistic.

The runner starts one server at a time, uses one Uvicorn worker and the same batch size,
waits for native compilation to finish, then runs Locust in a separate process. It saves
`/quote` throughput, p50/p95 latency, failures, CSVs, and logs under `locust-results/`,
then stops the server. The short runs below are demonstrations; use 30–60 second trials
for more dependable capacity measurements.

In [ ]:
API_BATCH_SIZE = int(os.environ.get('PRICING_COUNT', '200000'))
api_rows = []
if RUN_API_DEMO:
    import importlib
    import demo_runner
    # Pick up runner changes even when this notebook kernel was already running.
    importlib.reload(demo_runner)
    api_rows = demo_runner.run_suite(users=(1, 4), seconds=8, count=API_BATCH_SIZE, backend=API_BACKEND)
    from IPython.display import FileLink
    display(FileLink('locust-results/index.html', result_html_prefix='Latest HTML reports: '))
else:
    print('API runs disabled. Set RUN_API_DEMO = True or run python demo_runner.py.')

In [ ]:
if api_rows:
    lines = ['| Server | Backend | Users | Quotes | Failures | Quotes/s | p50 ms | p95 ms |',
             '|---|---|---:|---:|---:|---:|---:|---:|']
    for row in api_rows:
        lines.append(f'| {row["server"]} | {row["backend"]} | {row["users"]} | '
                     f'{row["requests"]} | {row["failures"]} | {row["requests_per_second"]:.1f} | '
                     f'{row["p50_ms"]:.1f} | {row["p95_ms"]:.1f} |')
    display(Markdown('\n'.join(lines)))

### Same input, different transport

With servers running using the README commands, a REST request is:

```python
import httpx
httpx.post('http://127.0.0.1:8001/quote',
           json={'discount': 0.1, 'tax_rate': 0.08, 'shipping_fee': 5}).json()
```

A WebSocket connection can receive another quote whenever the discount changes:

```python
import json
from websockets.asyncio.client import connect

async with connect('ws://127.0.0.1:8003/quote') as socket:
    for discount in (0.05, 0.1, 0.2):
        await socket.send(json.dumps({'discount': discount, 'tax_rate': 0.08, 'shipping_fee': 5}))
        quote = json.loads(await socket.recv())
        print(discount, quote['total_amount'])
```

The business logic is unchanged. WebSocket provides an ongoing bidirectional session;
it does not make the pricing calculation faster. This demo returns one response for each
message. Invalid REST bodies return HTTP 422; invalid WebSocket messages return an error
object and leave the connection ready for corrected settings.

### Experiments

1. Change `API_BACKEND` to `'numpy'`, then `'cuda'`. Does the fastest calculation also
   give the best API latency for this batch size?
2. Increase concurrent users from 1 to 4 to 16. Compare p95 latency and failures as well
   as throughput. The local load generator can also become a bottleneck.
3. Diff each REST/WebSocket pair. Identify the small transport change and the unchanged
   pricing code.
4. Increase the number of orders. Explain the results using Python loop overhead,
   compiled arithmetic, memory traffic, GPU transfers, and request overhead.